
# ML Algorithms

(`Dar Al-Ibda Al-Taqni | CYSTECH`)  

<div dir="rtl">

**المدربة:** **Eng. Tasneem Al'araj** ([LinkedIn](https://www.linkedin.com/in/tasneemalaraj))

---
### 🎯 محتوى هذا النوتبوك :
1. **تحميل البيانات النظيفة (`real_estate_cleaned.csv`)** وتحديد `X` و `y` مع منع تسريب البيانات (`Data Leakage`).
2. **الترميز (`One-Hot Encoding`)** للأعمدة النصية `Urban_Zone` و `Location_Tier`.
3. **التقسيم الثلاثي (`Train 70%` / `Validation 15%` / `Test 15%`)** ثم توحيد المقياس (`StandardScaler`) بناءً على `Train` فقط.
4. **تدريب ومقارنة نماذج الانحدار (`LinearRegression` & `RandomForestRegressor`)** ونموذج التصنيف (`LogisticRegression`) على مجموعة `Validation` ثم الاختبار النهائي على `Test`.
5. ** (`Interactive Prediction Form`):** مربعات تعبئة تفاعلية لمواصفات البيت يقوم المستخدم بتعبئتها ليتنبأ النموذج بالسعر المتوقع والفئة فوراً!

</div>


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    r2_score, mean_absolute_error, root_mean_squared_error,
    accuracy_score, confusion_matrix, classification_report
)

sns.set_theme(style='whitegrid')

csv_file = 'real_estate_cleaned.csv'
if not os.path.exists(csv_file):
    try:
        from google.colab import files
        print('يرجى رفع ملف البيانات  (real_estate_cleaned.csv):')
        uploaded = files.upload()
        csv_file = list(uploaded.keys())[0]
    except ImportError:
        pass

df = pd.read_csv(csv_file)
print(f'Loaded Dataset: {csv_file} | Shape: {df.shape}')
df.head()

<div dir="rtl">

# الخطوة 1: تحديد المدخلات `X` والهدف `y` وتطبيق `One-Hot Encoding`
- نستبعد `ID` (معرّف تسلسلي) و`Price_Per_SqM` (محسوب من السعر نفسه $\rightarrow$ تسريب بيانات!) و`Square_Feet_Converted` (مكرر).
- نحوّل الأعمدة النصية (`Urban_Zone`, `Location_Tier`) إلى أرقام ثنائية `0 / 1` باستخدام `pd.get_dummies(..., drop_first=True)`.
</div>


In [ ]:
feature_cols = [
    'Square_Meters', 'Num_Bedrooms', 'Num_Bathrooms', 'Num_Floors',
    'Year_Built', 'Has_Garden', 'Has_Pool', 'Garage_Area_SqM',
    'Location_Score', 'Urban_Zone', 'Location_Tier'
]
X = df[feature_cols].copy()
y_reg = df['Price']  # هدف الانحدار (توقع السعر الرقمي)

median_price = df['Price'].median()
y_clf = (df['Price'] > median_price).astype(int)  # هدف التصنيف (1 = عقار فخم، 0 = عادي)
print(f'Median Price Threshold for Is_Luxury: ${median_price:,.2f}')

# One-Hot Encoding
X_encoded = pd.get_dummies(X, columns=['Urban_Zone', 'Location_Tier'], drop_first=True, dtype=int)
print('Encoded Features Shape:', X_encoded.shape)
print('Feature Columns:', list(X_encoded.columns))

<div dir="rtl">

# الخطوة 2: تقسيم البيانات (`70% Train` / `15% Val` / `15% Test`) وتوحيد المقياس (`Rescaling`)
1. **`Train (70% = 350 صفاً)`:** لتدريب النموذج وتعلّم الأوزان.
2. **`Validation (15% = 75 صفاً)`:** لمقارنة النماذج (`LinearRegression` مقابل `RandomForestRegressor`) وضبط الإعدادات دون لمس الاختبار.
3. **`Test (15% = 75 صفاً)`:** امتحان نهائي مخفي يُستخدم مرة واحدة فقط لقياس الدقة الحقيقية.
4. **`StandardScaler`:** يُدرّب (`fit`) على `X_train` فقط ثم يُطبّق (`transform`) على `Validation` و `Test` لمنع تسريب البيانات.

</div>


In [ ]:
X_train, X_temp, y_reg_train, y_reg_temp, y_clf_train, y_clf_temp = train_test_split(
    X_encoded, y_reg, y_clf, test_size=0.30, random_state=42
)

X_val, X_test, y_reg_val, y_reg_test, y_clf_val, y_clf_test = train_test_split(
    X_temp, y_reg_temp, y_clf_temp, test_size=0.50, random_state=42
)

print(f'Train: {len(X_train)} rows (70%) | Validation: {len(X_val)} rows (15%) | Test: {len(X_test)} rows (15%)')

# توحيد المقياس (Feature Scaling): fit على Train فقط!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

#z-score=(x-mean)/std

<div dir="rtl">

# الخطوة 3: تدريب ومقارنة نماذج الانحدار (`Regression`) على `Val dataset` ثم التقييم النهائي على `Test`
</div>


In [ ]:
# 1. تدريب Linear Regression و Random Forest Regressor على مجموعة التدريب
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_reg_train)

rf_model = RandomForestRegressor(n_estimators=150, max_depth=10, random_state=42)
rf_model.fit(X_train_scaled, y_reg_train)

# 2. المقارنة على مجموعة التحقق
val_pred_lr = lr_model.predict(X_val_scaled)
val_pred_rf = rf_model.predict(X_val_scaled)

print('=== Validation Set Comparison (75 Houses) ===')
print(f'Linear Regression -> R²: {r2_score(y_reg_val, val_pred_lr):.3f} | MAE: ${mean_absolute_error(y_reg_val, val_pred_lr):,.0f} | RMSE: ${root_mean_squared_error(y_reg_val, val_pred_lr):,.0f}')
print(f'Random Forest     -> R²: {r2_score(y_reg_val, val_pred_rf):.3f} | MAE: ${mean_absolute_error(y_reg_val, val_pred_rf):,.0f} | RMSE: ${root_mean_squared_error(y_reg_val, val_pred_rf):,.0f}')

# اختيار النموذج الأفضل (Linear Regression تفوق في العلاقة الخطية) واختباره على Test Set
test_pred_lr = lr_model.predict(X_test_scaled)
print('\n=== Final Unbiased Evaluation on Test Set (75 Hidden Houses) ===')
print(f'Linear Regression Test -> R²: {r2_score(y_reg_test, test_pred_lr):.3f} | MAE: ${mean_absolute_error(y_reg_test, test_pred_lr):,.0f} | RMSE: ${root_mean_squared_error(y_reg_test, test_pred_lr):,.0f}')

# رسم القيم الحقيقية مقابل المتوقعة على مجموعة Test
plt.figure(figsize=(8, 5))
plt.scatter(y_reg_test / 1000, test_pred_lr / 1000, alpha=0.70, color='#0D9488', edgecolor='#0F172A', s=50)
plt.plot([250, 920], [250, 920], 'r--', linewidth=2, label='Perfect Prediction (y = x)')
plt.title(f'Test Set: Actual vs. Predicted House Price (R² = {r2_score(y_reg_test, test_pred_lr):.3f})')
plt.xlabel('Actual Price ($k)')
plt.ylabel('Predicted Price ($k)')
plt.legend()
plt.show()

<div dir="rtl">

# الخطوة 4: تدريب وتقييم نموذج التصنيف (`LogisticRegression`) على `Validation` و `Test`
</div>

In [ ]:
clf_model = LogisticRegression(random_state=42)
clf_model.fit(X_train_scaled, y_clf_train)

val_clf_pred  = clf_model.predict(X_val_scaled)
test_clf_pred = clf_model.predict(X_test_scaled)

print(f'Validation Accuracy: {accuracy_score(y_clf_val, val_clf_pred)*100:.1f}%')
print(f'Final Test Accuracy: {accuracy_score(y_clf_test, test_clf_pred)*100:.1f}%\n')
print(classification_report(y_clf_test, test_clf_pred, target_names=['Standard (0)', 'Luxury (1)']))

cm = confusion_matrix(y_clf_test, test_clf_pred)
plt.figure(figsize=(5.5, 4.2))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', cbar=False,
            xticklabels=['Standard (0)', 'Luxury (1)'],
            yticklabels=['Standard (0)', 'Luxury (1)'])
plt.title('Test Set Confusion Matrix: Is_Luxury')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


# 🏡 Interactive House Price Predictor
<div dir="rtl">

 النموذج يتنبأ **بالسعر المتوقع بالدولار** وتصنيف البيت (`Luxury` أم  `Standard`)!

</div>


In [ ]:
#@title 🏡 أدخل مواصفات البيت للتنبؤ بسعره وتصنيفه { run: "auto" }
Square_Meters = 65 #@param {type:"slider", min:50, max:350, step:5}
Num_Bedrooms = 2 #@param {type:"integer"}
Num_Bathrooms = 1 #@param {type:"integer"}
Num_Floors = 1 #@param {type:"integer"}
Year_Built = 1984 #@param {type:"slider", min:1900, max:2026, step:1}
Has_Garden = "Yes (1)" #@param ["No (0)", "Yes (1)"]
Has_Pool = "No (0)" #@param ["No (0)", "Yes (1)"]
Garage_Area_SqM = 15 #@param {type:"slider", min:10, max:50, step:1}
Location_Score = 3 #@param {type:"number"}
Urban_Zone = "Outskirts (>12km)" #@param ["Urban_Core (<5km)", "Suburban (5-12km)", "Outskirts (>12km)"]
Location_Tier = "Low_Score (0-3.5)" #@param ["Low_Score (0-3.5)", "Mid_Score (3.5-7)", "Prime_Score (7-10)"]

# تحويل الاختيارات النصية لـ Has_Garden و Has_Pool إلى أرقام 1 / 0
garden_val = 1 if '1' in Has_Garden else 0
pool_val   = 1 if '1' in Has_Pool else 0

# 1. بناء DataFrame من مدخلات المستخدم
user_house = pd.DataFrame([{
    'Square_Meters': Square_Meters,
    'Num_Bedrooms': Num_Bedrooms,
    'Num_Bathrooms': Num_Bathrooms,
    'Num_Floors': Num_Floors,
    'Year_Built': Year_Built,
    'Has_Garden': garden_val,
    'Has_Pool': pool_val,
    'Garage_Area_SqM': Garage_Area_SqM,
    'Location_Score': Location_Score,
    'Urban_Zone': Urban_Zone,
    'Location_Tier': Location_Tier
}])

# 2. تطبيق نفس الترميز (One-Hot Encoding) وضمان مطابقة أسماء وترتيب الأعمدة مع X_encoded
user_encoded = pd.get_dummies(user_house, columns=['Urban_Zone', 'Location_Tier'], dtype=int)
user_encoded = user_encoded.reindex(columns=X_encoded.columns, fill_value=0)

# 3. توحيد المقياس بنفس الـ Scaler المدرّب على Train
user_scaled = scaler.transform(user_encoded)

# 4. التنبؤ بالسعر والتصنيف
pred_lr_price = lr_model.predict(user_scaled)[0]
pred_rf_price = rf_model.predict(user_scaled)[0]
pred_class    = clf_model.predict(user_scaled)[0]
pred_prob     = clf_model.predict_proba(user_scaled)[0][1] * 100

print('=' * 55)
print(f'🏠 السعر المتوقع (Linear Regression): ${pred_lr_price:,.2f}')
print(f'🌲 السعر المتوقع (Random Forest)    : ${pred_rf_price:,.2f}')
tier_label = '🌟 عقار فخم (Luxury)' if pred_class == 1 else '🏘️ عقار عادي (Standard)'
print(f'🏷️ التصنيف المتوقع: {tier_label} (احتمالية الفخامة: {pred_prob:.1f}%)')
print('=' * 55)

<div dir="rtl">

# الخطوة 5: حفظ حزمة النموذج (`Model Bundle`) للتشغيل المحلي
- نحفظ النماذج المدرّبة (`lr_model`, `rf_model`, `clf_model`) مع `scaler` وترتيب الأعمدة `X_encoded.columns` ووسيط السعر `median_price` في ملف واحد (`real_estate_bundle.joblib`) باستخدام `joblib`.
</div>


In [ ]:
import joblib

model_bundle = {
    "reg_model": lr_model,
    "rf_model": rf_model,
    "clf_model": clf_model,
    "scaler": scaler,
    "feature_columns": list(X_encoded.columns),
    "median_threshold": float(median_price)
}

bundle_path = "real_estate_bundle.joblib"
joblib.dump(model_bundle, bundle_path)
print(f"✅ تم حفظ حزمة النموذج بنجاح في: {bundle_path} ({os.path.getsize(bundle_path):,} bytes)")

# تنزيل الملف تلقائياً إلى جهازك في Google Colab
try:
    from google.colab import files
    files.download(bundle_path)
except ImportError:
    print("الملف جاهز في المجلد الحالي.")
